# **Task 3 — Modeling (N-gram Language Models)**  
### SwiftKey Data Science Capstone  
**Author:** Sonal Kumari

This notebook builds the first predictive text model using:
- Unigrams (1-gram)
- Bigrams (2-gram)
- Trigrams (3-gram)
- 4-grams (optional extension)
- Smoothing (Add-k)
- Backoff model (Stupid Backoff)

We will train the n-gram dictionaries and then build a *next word prediction* function.


### 🔹 Load the tokenized files from Task 1  
We will use these to build n-gram models.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import math

tokenized_dir = Path("data/tokenized")

blogs = tokenized_dir / "blogs_tokenized.txt"
news = tokenized_dir / "news_tokenized.txt"
twitter = tokenized_dir / "twitter_tokenized.txt"

files = [blogs, news, twitter]


### 🔹 Function to Generate N-grams  
We build n-grams for n = 1, 2, 3 using an efficient streaming approach.


In [3]:
def generate_ngrams(path, n=2, max_lines=None):
    counter = Counter()
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            tokens = line.split()
            if len(tokens) < n:
                continue
            for j in range(len(tokens)-n+1):
                gram = tuple(tokens[j:j+n])
                counter[gram] += 1
    return counter

print("Building n-grams ...")

unigrams = Counter()
for file in files:
    unigrams.update(generate_ngrams(file, n=1))

bigrams = Counter()
for file in files:
    bigrams.update(generate_ngrams(file, n=2))

trigrams = Counter()
for file in files:
    trigrams.update(generate_ngrams(file, n=3))

print("Done! N-grams built.")


Building n-grams ...
Done! N-grams built.


### 🔹 Smoothing: Add-k (Laplace)  
We use Add-k smoothing to avoid zero-probabilities:

P(w_i | previous_n_words) = (count + k) / (total + k * V)

Where:  
- k = small constant (commonly 0.5 or 1)  
- V = vocabulary size  


In [5]:
V = len(unigrams)
k = 0.5  # smoothing constant

def smoothed_prob(ngram_count, history_count):
    return (ngram_count + k) / (history_count + k * V)


### 🔹 Build prefix → next word lookup tables  
For example:
- For bigrams:  ("i",) → {"am": 200, "have": 130, ...}  
- For trigrams: ("i", "am") → {"happy": 40, "going": 15, ...}  


In [6]:
from collections import defaultdict

def build_lookup(counter, n):
    lookup = defaultdict(Counter)
    for gram, count in counter.items():
        history = gram[:-1]
        next_word = gram[-1]
        lookup[history][next_word] += count
    return lookup

bigram_lookup = build_lookup(bigrams, 2)
trigram_lookup = build_lookup(trigrams, 3)


# 🔹 Stupid Backoff
If a trigram is unseen → look for bigram  
If bigram is unseen → use unigram

Score = α * probability of lower order model  
where α ≈ 0.4


In [7]:
alpha = 0.4

def predict_next_word(text):
    tokens = text.lower().split()
    if len(tokens) >= 2:
        w1, w2 = tokens[-2:]
        # Try trigram
        if (w1, w2) in trigram_lookup:
            return trigram_lookup[(w1, w2)].most_common(5)
    if len(tokens) >= 1:
        w1 = tokens[-1]
        # Try bigram
        if (w1,) in bigram_lookup:
            return bigram_lookup[(w1,)].most_common(5)
    # Fallback: unigrams
    return unigrams.most_common(5)


### 🔹 Try predictions with natural text examples


In [8]:
tests = [
    "i love",
    "how are",
    "can you",
    "i want to",
    "we are going"
]

for t in tests:
    print("\nInput:", t)
    print("Predictions:", predict_next_word(t))



Input: i love
Predictions: [('you', 48), ('the', 23), ('it', 16), ('to', 12), ('that', 10)]

Input: how are
Predictions: [('you', 21), ('things', 2), ('the', 1), ('genetically', 1), ('those', 1)]

Input: can you
Predictions: [('please', 7), ('not', 3), ('believe', 3), ('tell', 3), ('help', 3)]

Input: i want to
Predictions: [('be', 31), ('go', 27), ('do', 21), ('get', 13), ('know', 12)]

Input: we are going
Predictions: [('to', 29), ('if', 1), ('during', 1), ('down', 1), ('into', 1)]


# 🔹 Storage Efficiency

We store:
- Only the lookup dictionaries (prefix → next-word counts)
- Compressed using `joblib` for fast loading
- They will be used by the prediction engine in Task 4

This keeps memory low and runtime fast.


In [9]:
import joblib

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

joblib.dump(bigram_lookup, model_dir/"bigram_lookup.joblib")
joblib.dump(trigram_lookup, model_dir/"trigram_lookup.joblib")
joblib.dump(unigrams, model_dir/"unigrams.joblib")

print("Models saved successfully!")


Models saved successfully!


### 🔹 Perplexity (optional)
Measures how well the model predicts test data.
Lower = better.

We can compute basic perplexity for trigrams.


In [10]:
def trigram_perplexity(counter):
    total_log_prob = 0
    N = sum(counter.values())

    for gram, count in counter.items():
        w1, w2, w3 = gram
        history = (w1, w2)
        history_count = sum(trigram_lookup[history].values())
        prob = smoothed_prob(count, history_count)
        total_log_prob += count * math.log(prob)

    return math.exp(-total_log_prob / N)

print("Approx Perplexity:", trigram_perplexity(trigrams))


Approx Perplexity: 9660.810133299874


# ✔ Task 3 — Answers to Key Questions

### **1️⃣ How can you efficiently store an n-gram model?**
Use prefix → next word lookup tables:
- bigram_lookup[(“i”)] → {"am": 200, "have": 130}
- trigram_lookup[(“i”, “am”)] → {"happy": 40}

They compress well and load quickly.

---

### **2️⃣ How can word frequencies help efficiency?**
- Remove words that occur < 2 or < 3 times
- They contribute noise and increase memory
- Reduces model size drastically (~50–80%)

---

### **3️⃣ How big should n be?**
- n = 3 (trigram) is standard
- n = 4 improves accuracy but increases memory
- Backoff makes trigram sufficient

---

### **4️⃣ How to smooth probabilities?**
Common methods:
- Add-k (Laplace smoothing)
- Good-Turing
- Kneser-Ney (best for language models)

---

### **5️⃣ How to evaluate the model?**
- Perplexity
- Prediction accuracy on held-out text
- Runtime per prediction (< 100 ms target)

---

### **6️⃣ How to estimate probability of unseen n-grams?**
Use **backoff**:
- If trigram unseen → bigram
- If bigram unseen → unigram
- Apply discount α ≈ 0.4


# 🎉 Task 3 Completed Successfully!

We have now:
✔ Built unigram, bigram, trigram models  
✔ Implemented smoothing  
✔ Built a Stupid Backoff predictor  
✔ Saved model files  
✔ Evaluated model performance  

Next Step → **Task 4: Prediction Engine**  
This will power our final Shiny/Flask App.

